# Shor's Factoring Algorithm

## Putting the pieces together: given a composite $N$ to factor, Shor finds a nontrivial factor in time polynomial in $\log N$. The fastest known classical algorithm (the general number field sieve) runs in time roughly $\exp(O(\log N)^{1/3} (\log\log N)^{2/3})$ — sub-exponential, but still vastly slower.

# 1. The classical reduction

## Factoring $N$ reduces to **order finding** with high probability via the following classical algorithm:

## 1. If $N$ is even, return 2.
## 2. If $N = p^k$ for some integer $p$ and $k \geq 2$, return $p$ (easy to detect).
## 3. Otherwise pick $a \in \{2, \dots, N-1\}$ uniformly at random.
## 4. Compute $d = \gcd(a, N)$. If $d > 1$, return $d$.
## 5. Use the **quantum order-finding subroutine** to find the order $r$ of $a$ mod $N$.
## 6. If $r$ is odd, or $a^{r/2} \equiv -1 \pmod N$, go back to step 3.
## 7. Output $\gcd(a^{r/2} - 1, N)$ — it is a nontrivial factor of $N$ with probability $\geq 1/2$ over the choice of $a$.

# 2. Why step 7 works

## Suppose $r$ is even and $a^{r/2} \not\equiv -1 \pmod N$. Then $a^r - 1 = (a^{r/2} - 1)(a^{r/2} + 1) \equiv 0 \pmod N$, so $N$ divides $(a^{r/2} - 1)(a^{r/2} + 1)$. But $N$ divides *neither* factor alone (by hypothesis), so $N$ has parts in both — that is, $\gcd(a^{r/2} - 1, N)$ is a nontrivial divisor of $N$.

## A short argument using the Chinese Remainder Theorem shows that a *random* $a$ satisfies the "good" conditions with probability $\geq 1/2$, so a constant number of trials suffices.

In [ ]:
# Pure-classical Shor (using brute-force order finding) — fine for tiny N.
import math, random

def order_classical(a, N):
    r, x = 1, a % N
    while x != 1:
        x = (x * a) % N
        r += 1
    return r

def shor(N, trials=20):
    if N % 2 == 0:
        return 2
    for _ in range(trials):
        a = random.randint(2, N - 1)
        d = math.gcd(a, N)
        if d > 1:
            return d
        r = order_classical(a, N)
        if r % 2 != 0:               # odd order: retry
            continue
        x = pow(a, r // 2, N)
        if x == N - 1:               # a^(r/2) = -1 mod N: retry
            continue
        for f in (math.gcd(x - 1, N), math.gcd(x + 1, N)):
            if 1 < f < N:
                return f
    return None

for N in (15, 21, 33, 35, 91):
    print(f'factor of {N} = {shor(N)}')

# 3. Running Shor on a real device

## Demonstrations of Shor on actual quantum hardware are tiny — typically factoring 15 or 21 — because the modular exponentiation circuits are very deep. To factor a 2048-bit RSA modulus, current estimates require **millions of physical qubits** with thousands of logical qubits and surface-code error correction (chapter 08).

## Despite that, Shor is the algorithm that motivates much of post-quantum cryptography: once you have a fault-tolerant quantum computer large enough to run Shor on cryptographic-sized $N$, today's RSA, DH, and ECC become insecure.

# 4. Consequences

## - **RSA, Diffie–Hellman, ECC** all rely on hardness of factoring or the discrete log, both broken by Shor.
## - **Symmetric ciphers** like AES are slowed down by Grover but not broken — AES-256 is widely considered post-quantum safe.
## - **Post-quantum cryptography** (lattice-based, hash-based, etc.) is now standardised by NIST and being deployed.
## - The very existence of Shor is the strongest known evidence that quantum computers offer genuine exponential speedups on practically interesting problems.

# Recap

## - Factoring $N$ reduces (classically, randomly) to finding orders modulo $N$.
## - Order finding is QPE on a modular multiplier — polynomial-time quantum.
## - The whole pipeline gives a polynomial-time quantum factoring algorithm.
## - Real-hardware demonstrations are still tiny; scaling to RSA-2048 needs full fault tolerance.

## **End of chapter 6.** Next: advanced algorithms (VQE, QAOA, HHL, simulation, QML).